# Vector stores and semantic search



https://github.com/ekohrt/animal-fun-facts-dataset/blob/main/animal-fun-facts-dataset.csv

https://github.com/walkerkq/musiclyrics/tree/master

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\spisi\Documents\repos\computer-inteligence\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part I: Basic vector store implementation

In [3]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        document_texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(document_texts)
        self.embeddings.extend(new_embeddings)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embedding = self.embedding_model.encode([query])[0]
        scores = cosine_similarity([query_embedding], self.embeddings)[0]
        top_docs = np.argsort(scores)[::-1][:top_k]

        return [SearchResult(scores[i], self.documents[i]) for i in top_docs]


In [4]:
import pandas as pd
url = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"
df = pd.read_csv(url)
print(df.head())

  animal_name                                             source  \
0    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
1    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
2    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
3    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
4    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   

                                                text media_link  \
0  Aardvarks are sometimes called "ant bears", "e...        NaN   
1  Aardvarks\nhave rather primitive brains that a...        NaN   
2  Aardvarks\nteeth are lined with fine upright t...        NaN   
3  The aardvarks Latin family name "Tubulidentata...        NaN   
4  Baby aardvarks are born with front teeth that ...        NaN   

   wikipedia_link  
0  /wiki/Aardvark  
1  /wiki/Aardvark  
2  /wiki/Aardvark  
3  /wiki/Aardvark  
4  /wiki/Aardvark  


In [9]:
# Create document instances
documents = []

for i, row in df.iterrows():
    doc = Document(
        text=str(row["text"]), 
        metadata={
            "animal_name": str(row["animal_name"]),
            "source": str(row["source"]),
            "media_link": str(row["media_link"]),
            "wikipedia_link": str(row["wikipedia_link"])
            })
    documents.append(doc)
print(f"Total documents: {len(documents)}")

Total documents: 7734


In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')
vector_store = VectorStore(model)
vector_store.add_documents(documents)

queries = [
    "Which animals are similar to humans?",
    "Which animals can fly, walk and swim?",
    "Which animals eat berries?",
    "Animals that sustain extreme cold temperatures",
    "What are the smallest monkeys?"
]

for query in queries:
    print(f"\nQuery: {query}")
    results = vector_store.search(query, top_k=3)
    for result in results:
        print(f"Score: {result.score:.4f}, Animal: {result.document.metadata['animal_name']}, \nText: {result.document.text}\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2382.13it/s]



Query: Which animals are similar to humans?
Score: 0.5825, Animal: tapir, 
Text: Most closely related to horses and rhinos!

Score: 0.5666, Animal: bird of paradise, 
Text: There are around 50 different species!

Score: 0.5661, Animal: bonobo, 
Text: Bonobos and chimps were once thought to be the same animal


Query: Which animals can fly, walk and swim?
Score: 0.6654, Animal: colugo (flying lemur), 
Text: They don’t fly, they glide.
The only mammal which can independently fly is the bat. Instead, colugas glide which works in the same way as a wingsuit.

Score: 0.6602, Animal: lamprey, 
Text: They technically can’t swim.
Despite living in water, their propulsion methods aren’t traditional.

Score: 0.6548, Animal: komodo dragon, 
Text: They can also swim.
Not only do they move fast on land, but they are excellent swimmers.


Query: Which animals eat berries?
Score: 0.6601, Animal: arctic hare, 
Text: Eats berries found in the snow!

Score: 0.6181, Animal: fainting goat, 
Text: They are

## Part II: Filtering by metadata

In [47]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        document_texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(document_texts)
        self.embeddings.extend(new_embeddings)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        query_embedding = self.embedding_model.encode([query])
        scores = cosine_similarity(query_embedding, self.embeddings)[0]
        top_indices = scores.argsort()[::-1][:top_k]

        filtered_docs = []
        filtered_embeddings = []

        for doc, emb in zip(self.documents, self.embeddings):
            include_doc = True

            if metadata_filter is not None:
                for key, value in metadata_filter.items():
                    doc_value = doc.metadata.get(key)
                    
                    # Support range for years
                    if isinstance(value, (tuple, list)) and len(value) == 2:
                        try:
                            doc_num = float(doc_value)
                            min_val, max_val = float(value[0]), float(value[1])
                            if not (min_val <= doc_num <= max_val):
                                include_doc = False
                                break
                        except (ValueError, TypeError):
                            include_doc = False
                            break
                    # contains string and lower for artist and song fields
                    elif key in ['artist', 'song']:
                        if not (isinstance(value, str) and isinstance(doc_value, str) and 
                                value.lower() in doc_value.lower()):
                            include_doc = False
                            break
                    # Exact match for other fields
                    elif doc_value != value:
                        include_doc = False
                        break

            if include_doc:
                filtered_docs.append(doc)
                filtered_embeddings.append(emb)

        if len(filtered_docs) == 0:
            return []
        
        scores = cosine_similarity(query_embedding, filtered_embeddings)[0]
        top_indices = scores.argsort()[::-1][:top_k]

        return [SearchResult(scores[i], filtered_docs[i]) for i in top_indices]

In [44]:
url = "https://raw.githubusercontent.com/walkerkq/musiclyrics/master/billboard_lyrics_1964-2015.csv"

# non UTF-8 characters in dataset
df = pd.read_csv(url, encoding='latin-1')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

Dataset shape: (5100, 6)
Columns: ['Rank', 'Song', 'Artist', 'Year', 'Lyrics', 'Source']
   Rank                                      Song  \
0     1                               wooly bully   
1     2  i cant help myself sugar pie honey bunch   
2     3                i cant get no satisfaction   
3     4                       you were on my mind   
4     5              youve lost that lovin feelin   

                          Artist  Year  \
0  sam the sham and the pharaohs  1965   
1                      four tops  1965   
2             the rolling stones  1965   
3                        we five  1965   
4         the righteous brothers  1965   

                                              Lyrics  Source  
0  sam the sham miscellaneous wooly bully wooly b...     3.0  
1   sugar pie honey bunch you know that i love yo...     1.0  
2                                                        1.0  
3   when i woke up this morning you were on my mi...     1.0  
4   you never close your

In [ ]:
documents = []

for _, row in df.iterrows():
    doc = Document(
        text=str(row["Lyrics"])[:400],
        metadata={
            "song": str(row["Song"]),
            "artist": str(row["Artist"]),
            "year": str(row["Year"]),
            "rank": str(int(row["Rank"])),
            "source": str(row["Source"])
        }
    )
    documents.append(doc)

print(f"{len(documents)}")
ex = documents[0]
print(f"\nExample — {ex.metadata['song']} by {ex.metadata['artist']} ({ex.metadata['year']})")
print(f"Rank: #{ex.metadata['rank']}")
print(f"Lyrics: {ex.text[:150]}...")

5100

Example — wooly bully by sam the sham and the pharaohs (1965)
Source: 3.0  |  Rank: #1
Lyrics: sam the sham miscellaneous wooly bully wooly bully sam the sham  the pharaohs  domingo samudio uno dos one two tres quatro matty told hatty about a th...


In [49]:
model = SentenceTransformer('all-MiniLM-L6-v2')
filtered_vector_store = FilteredVectorStore(model)
filtered_vector_store.add_documents(documents)

queries = [
    ("california dreamin", {"year": "1966"}),
    ("dancing party celebration having fun tonight", {"year": "2000"}),
    ("freedom equality social change protest", {"year": "1969"}),
    ("california", {"artist": "The Beach Boys"}),
    ("nice to meet you", {"rank": "7", "artist": "Taylor Swift"}),
    ("romantic love forever together devoted", {"year": ("1990", "2000")}),
    ("love song", {"year": ("1980", "1990")})
]

for query, f in queries:
    print(f"\nQuery: '{query}'")
    print(f"Filter: {f}")
    results = filtered_vector_store.search(query, top_k=3, metadata_filter=f)
    for result in results:
        m = result.document.metadata
        print(f"- Score: {result.score:.4f} | {m['song']} by {m['artist']} ({m['year']}) | Rank: #{m['rank']}")
        print(f"- Lyrics: {result.document.text[:150].strip()}...")
        print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11287.68it/s]



Query: 'california dreamin'
Filter: {'year': '1966'}
- Score: 0.4553 | california dreamin by the mamas  the papas (1966) | Rank: #10
- Lyrics: all the leaves are brown and the sky is grey ive been for a walk on a winters day id be safe and warm if i was in la california dreamin on such a wint...

- Score: 0.3740 | daydream by the lovin spoonful (1966) | Rank: #41
- Lyrics: what a day for a daydream what a day for a daydreamin boy and im lost in a daydream dreamin bout my bundle of joy and even if time aint really on my s...

- Score: 0.3199 | you baby by the turtles (1966) | Rank: #87
- Lyrics: the turtles miscellaneous you baby you baby the turtles pf sloan  steve barri intro drum beat followed by 12string guitar b dm e b v v v v v v v v v v...


Query: 'dancing party celebration having fun tonight'
Filter: {'year': '2000'}
- Score: 0.3861 | dance with me by debelah morgan (2000) | Rank: #89
- Lyrics: oh come and dance with me my baby lets dance till we go crazy the night is young an